# 03 Agent 循环

ReAct 模式：思考 → 行动 → 观察 → 再思考，LLM 自己决定何时停下来。

In [ ]:
import sys, json
sys.path.insert(0, '..')

from src.client import get_client

client = get_client()

## 工具

In [ ]:
def get_weather(city):
    w = {"北京": {"temp": 25, "desc": "晴朗"}, "上海": {"temp": 28, "desc": "多云"},
         "东京": {"temp": 22, "desc": "小雨"}, "纽约": {"temp": 15, "desc": "阴天"}}
    for k in w:
        if k in city:
            return json.dumps(w[k], ensure_ascii=False)
    return json.dumps({"temp": 20, "desc": "未知"}, ensure_ascii=False)

def search_web(query):
    results = {
        "特斯拉": "特斯拉当前股价 $245，上季度为 $220。",
        "茅台": "茅台当前股价 ¥1650。",
        "图灵奖": "图灵奖是计算机领域最高荣誉，由ACM于1966年设立。",
        "2024奥运会": "2024年夏季奥运会在法国巴黎举办。",
        "东京人口": "东京都人口约1400万。",
    }
    for k, v in results.items():
        if k in query or query in k:
            return v
    return f"未找到关于'{query}'的结果。"

def calculate(expression):
    try:
        return str(eval(expression))
    except:
        return "计算错误"

tool_map = {"get_weather": get_weather, "search_web": search_web, "calculate": calculate}

tools = [
    {"type": "function", "function": {"name": "get_weather", "description": "查询指定城市的天气",
        "parameters": {"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]}}},
    {"type": "function", "function": {"name": "search_web", "description": "搜索网页获取知识或信息",
        "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}},
    {"type": "function", "function": {"name": "calculate", "description": "执行数学计算",
        "parameters": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]}}},
]

## ReAct 循环

In [ ]:
def agent(query, max_rounds=10):
    messages = [{"role": "user", "content": query}]

    for round_num in range(1, max_rounds + 1):
        response = client.chat.completions.create(
            model="deepseek-chat", messages=messages, tools=tools,
        )
        msg = response.choices[0].message

        if msg.tool_calls:
            print(f"--- Round {round_num} ---")
            messages.append(msg)
            for tc in msg.tool_calls:
                name = tc.function.name
                args = json.loads(tc.function.arguments)
                result = tool_map[name](**args)
                print(f"  [{name}] {args} → {result}")
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
        else:
            print(f"--- Round {round_num} (最终) ---")
            print(f"回答: {msg.content}")
            return

    print(f"达到最大轮次 {max_rounds}，强制退出。")

agent("北京和上海今天天气怎么样？")
print("\n" + "="*50 + "\n")
agent("特斯拉股价涨了多少？计算涨幅百分比。")
print("\n" + "="*50 + "\n")
agent("东京天气怎么样？如果适合出门的话帮我查一下东京的人口。")